Read Bronze Customers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.customers"
)

Cast source strings to business data types

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "date_of_birth_cast",
        F.to_date("date_of_birth", "yyyy-MM-dd")
    )
    .withColumn(
        "signup_date_cast",
        F.to_date("signup_date", "yyyy-MM-dd")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at", "yyyy-MM-dd HH:mm:ss")
    )
)

Standardize text fields

In [0]:
standardized_df = (
    typed_df
    .withColumn("first_name_clean", F.trim("first_name"))
    .withColumn("last_name_clean", F.trim("last_name"))
    .withColumn("email_clean", F.lower(F.trim("email")))
    .withColumn(
        "customer_status_clean",
        F.upper(F.trim("customer_status"))
    )
    .withColumn(
        "loyalty_tier_clean",
        F.upper(F.trim("loyalty_tier"))
    )
    .withColumn(
        "preferred_language_clean",
        F.upper(F.trim("preferred_language"))
    )
)

Define data-quality rules

In [0]:
validated_df = (
    standardized_df
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_invalid_email",
        ~F.col("email_clean").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("customer_status_clean").isin(
            "ACTIVE",
            "INACTIVE",
            "SUSPENDED"
        )
    )
    .withColumn(
        "dq_invalid_dob",
        F.col("date_of_birth_cast").isNull()
        | (F.col("date_of_birth_cast") > F.current_date())
    )
)

Critical failure flag

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_customer_id")
    | F.col("dq_invalid_email")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_dob")
)

Split valid and rejected records

In [0]:
reject_df = (
    validated_df
    .filter(F.col("dq_critical_failure"))
)

In [0]:
valid_df = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
)

above result like 

Bronze

   ↓

Validation

   ├── Valid   → Silver
   
   └── Invalid → Quarantine


In [0]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
    .filter(F.col("_row_number") == 1)
)

Build final Silver structure

In [0]:
silver_df = deduped_df.select(
    F.col("customer_id"),

    F.col("first_name_clean").alias("first_name"),
    F.col("last_name_clean").alias("last_name"),
    F.col("email_clean").alias("email"),

    "phone_number",

    F.col("date_of_birth_cast").alias("date_of_birth"),
    F.upper(F.trim("gender")).alias("gender"),

    F.col("signup_date_cast").alias("signup_date"),

    F.col("customer_status_clean").alias("customer_status"),
    F.col("loyalty_tier_clean").alias("loyalty_tier"),
    F.col("preferred_language_clean").alias("preferred_language"),

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias("_silver_processed_at")
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.customers"
    )
)

Write rejected records

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),
            F.when(
                F.col("dq_invalid_email"),
                F.lit("INVALID_EMAIL")
            ),
            F.when(
                F.col("dq_invalid_status"),
                F.lit("INVALID_CUSTOMER_STATUS")
            ),
            F.when(
                F.col("dq_invalid_dob"),
                F.lit("INVALID_DATE_OF_BIRTH")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.customer_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.customer_rejects ADD COLUMNS (_pipeline_run_id STRING)")

(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.customer_rejects"
    )
)

Verify counts

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.customers"
).count()

silver_count = spark.table(
    "customer360_dev.silver.customers"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.customer_rejects"
).count()

print(f"Bronze : {bronze_count:,}")
print(f"Silver : {silver_count:,}")
print(f"Reject : {reject_count:,}")

Bronze : 1,000,000
Silver : 986,000
Reject : 9,000
